# NB10 — Statistical Analysis of Documentation Gap Drivers

## Objective

This notebook formally tests which hospital characteristics predict documentation gap scores using:

1. **Hypothesis tests** — ANOVA, Kruskal-Wallis, and Mann-Whitney U tests for group differences
2. **Correlation analysis** — Spearman and Pearson correlations for continuous predictors
3. **OLS Regression** — Multivariate model to identify independent drivers of the documentation gap
4. **Logistic Regression** — Predict probability of being a "High" gap hospital

### Hypotheses

- **H1**: Private (for-profit) hospitals have smaller documentation gaps than public (government) hospitals, as they may be better resourced for revenue cycle operations.
- **H2**: Hospitals with higher provider-to-patient ratios have better documentation and smaller gaps than lower-resourced hospitals. We test this using staffing ratios (RN/bed, physician/bed, clinical staff/bed) from the CMS POS file.

### Data Sources

- **NB10a enriched features** (`hospital_enriched_features.csv`): Master dataset with staffing ratios from the CMS POS file and quality metrics from Hospital General Information
- **NB07 gap scores** (`hospital_gap_scores.csv`): Documentation gap score and tier classification

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load the enriched dataset from NB10a (has staffing ratios + quality metrics)
PROJECT_ROOT = Path.cwd().parents[1]

enriched_path = PROJECT_ROOT / 'data' / 'outputs' / 'nb10a_additional_features' / 'hospital_enriched_features.csv'

if enriched_path.exists():
    df = pd.read_csv(enriched_path, dtype={'ccn': str})
    print(f'Loaded enriched dataset from NB10a: {len(df):,} hospitals, {len(df.columns)} columns')
else:
    # Fallback: load NB06 benchmarks if NB10a hasn't been run yet
    print('WARNING: NB10a output not found. Falling back to NB06 benchmarks (no staffing/quality features).')
    df = pd.read_csv(
        PROJECT_ROOT / 'data' / 'outputs' / 'nb06_peer_benchmarks' / 'hospital_with_benchmarks.csv',
        dtype={'ccn': str}
    )

# Load NB07 gap scores
gap_scores = pd.read_csv(
    PROJECT_ROOT / 'data' / 'outputs' / 'nb07_gap_scores' / 'hospital_gap_scores.csv',
    dtype={'ccn': str}
)

# Merge gap scores into full dataset
df['ccn'] = df['ccn'].str.zfill(6)
gap_scores['ccn'] = gap_scores['ccn'].str.zfill(6)

score_cols = [c for c in gap_scores.columns if c not in df.columns]
score_cols.append('ccn')
df = df.merge(gap_scores[score_cols], on='ccn', how='left')

# Filter to hospitals with complete data for modeling
model_df = df.dropna(subset=['doc_gap_score', 'cmi', 'ownership_category']).copy()

print(f'Full dataset: {len(df):,} hospitals')
print(f'Modeling dataset (complete cases): {len(model_df):,} hospitals')
print(f'\nDoc gap score: mean={model_df["doc_gap_score"].mean():.2f}, std={model_df["doc_gap_score"].std():.2f}')
print(f'\nOwnership distribution:')
print(model_df['ownership_category'].value_counts())

# Check availability of enriched features
print(f'\n--- Staffing Features (from NB10a / POS file) ---')
staffing_cols = ['rn_per_bed', 'lpn_per_bed', 'physician_per_bed', 'total_nursing_per_bed',
                 'clinical_staff_per_bed', 'rn_skill_mix', 'total_clinical_staff']
for col in staffing_cols:
    if col in model_df.columns:
        n = model_df[col].notna().sum()
        print(f'  {col}: {n:,} ({100*n/len(model_df):.1f}%)')
    else:
        print(f'  {col}: NOT AVAILABLE')

print(f'\n--- Quality Features (from NB10a / Hospital General Info) ---')
quality_cols = ['star_rating', 'quality_score', 'mort_pct_worse', 'safety_pct_worse', 'readm_pct_worse']
for col in quality_cols:
    if col in model_df.columns:
        n = model_df[col].notna().sum()
        print(f'  {col}: {n:,} ({100*n/len(model_df):.1f}%)')
    else:
        print(f'  {col}: NOT AVAILABLE')

print(f'\n--- Original Features ---')
original_cols = ['resident_to_bed_ratio', 'beds', 'is_teaching', 'is_urban',
                 'wage_index', 'dsh_pct', 'drg_diversity']
for col in original_cols:
    if col in model_df.columns:
        n = model_df[col].notna().sum()
        print(f'  {col}: {n:,} ({100*n/len(model_df):.1f}%)')
    else:
        print(f'  {col}: NOT AVAILABLE')

Loaded enriched dataset from NB10a: 3,280 hospitals, 87 columns
Full dataset: 3,280 hospitals
Modeling dataset (complete cases): 3,060 hospitals

Doc gap score: mean=48.83, std=11.56

Ownership distribution:
ownership_category
Nonprofit     1925
For-Profit     629
Government     438
Other           68
Name: count, dtype: int64

--- Staffing Features (from NB10a / POS file) ---
  rn_per_bed: 3,060 (100.0%)
  lpn_per_bed: 3,060 (100.0%)
  physician_per_bed: 3,060 (100.0%)
  total_nursing_per_bed: 3,060 (100.0%)
  clinical_staff_per_bed: 3,060 (100.0%)
  rn_skill_mix: 3,003 (98.1%)
  total_clinical_staff: 3,060 (100.0%)

--- Quality Features (from NB10a / Hospital General Info) ---
  star_rating: 2,536 (82.9%)
  quality_score: 3,060 (100.0%)
  mort_pct_worse: 2,735 (89.4%)
  safety_pct_worse: 2,883 (94.2%)
  readm_pct_worse: 2,994 (97.8%)

--- Original Features ---
  resident_to_bed_ratio: 3,060 (100.0%)
  beds: 3,060 (100.0%)
  is_teaching: 3,060 (100.0%)
  is_urban: 3,060 (100.0%)
  wag

## Part 1: Hypothesis Testing — Ownership Type (H1)

**H1**: For-profit hospitals have smaller documentation gaps than government hospitals.

Tests:
- **Kruskal-Wallis H-test**: Non-parametric test for differences across all ownership groups
- **Mann-Whitney U test**: Pairwise test specifically comparing For-Profit vs. Government
- **One-way ANOVA**: Parametric alternative (included for completeness)
- **Effect size**: Cohen's d for the For-Profit vs. Government comparison

In [2]:
print('='*70)
print('HYPOTHESIS 1: OWNERSHIP TYPE AND DOCUMENTATION GAP')
print('='*70)

# Group statistics
print('\n--- Descriptive Statistics by Ownership ---')
ownership_stats = model_df.groupby('ownership_category')['doc_gap_score'].agg(
    ['count', 'mean', 'median', 'std']
).round(2)
ownership_stats.columns = ['N', 'Mean', 'Median', 'Std Dev']
print(ownership_stats.sort_values('Mean', ascending=False))

# Separate groups
groups = {name: group['doc_gap_score'].values 
          for name, group in model_df.groupby('ownership_category')}

# --- Test 1: Kruskal-Wallis (non-parametric ANOVA) ---
print('\n--- Kruskal-Wallis H-Test (all ownership groups) ---')
group_values = [g for g in groups.values() if len(g) >= 5]
h_stat, kw_p = stats.kruskal(*group_values)
print(f'H-statistic: {h_stat:.3f}')
print(f'p-value: {kw_p:.6f}')
print(f'Result: {"Significant" if kw_p < 0.05 else "Not significant"} at alpha=0.05')

# --- Test 2: One-way ANOVA (parametric) ---
print('\n--- One-Way ANOVA (all ownership groups) ---')
f_stat, anova_p = stats.f_oneway(*group_values)
print(f'F-statistic: {f_stat:.3f}')
print(f'p-value: {anova_p:.6f}')
print(f'Result: {"Significant" if anova_p < 0.05 else "Not significant"} at alpha=0.05')

# --- Test 3: Mann-Whitney U — For-Profit vs. Government (direct H1 test) ---
print('\n--- Mann-Whitney U Test: For-Profit vs. Government ---')
if 'For-Profit' in groups and 'Government' in groups:
    fp = groups['For-Profit']
    gov = groups['Government']
    u_stat, mw_p = stats.mannwhitneyu(fp, gov, alternative='less')  # H1: FP < Gov
    print(f'For-Profit mean: {fp.mean():.2f} (n={len(fp)})')
    print(f'Government mean: {gov.mean():.2f} (n={len(gov)})')
    print(f'U-statistic: {u_stat:.1f}')
    print(f'p-value (one-sided, FP < Gov): {mw_p:.6f}')
    print(f'Result: {"Significant" if mw_p < 0.05 else "Not significant"} at alpha=0.05')
    
    # Cohen's d effect size
    pooled_std = np.sqrt((fp.std()**2 + gov.std()**2) / 2)
    cohens_d = (gov.mean() - fp.mean()) / pooled_std
    print(f'Cohen\'s d: {cohens_d:.3f} ({"small" if abs(cohens_d) < 0.5 else "medium" if abs(cohens_d) < 0.8 else "large"} effect)')

# --- Test 4: Mann-Whitney U — For-Profit vs. Nonprofit ---
print('\n--- Mann-Whitney U Test: For-Profit vs. Nonprofit ---')
if 'For-Profit' in groups and 'Nonprofit' in groups:
    fp = groups['For-Profit']
    np_grp = groups['Nonprofit']
    u_stat2, mw_p2 = stats.mannwhitneyu(fp, np_grp, alternative='less')
    print(f'For-Profit mean: {fp.mean():.2f} (n={len(fp)})')
    print(f'Nonprofit mean: {np_grp.mean():.2f} (n={len(np_grp)})')
    print(f'U-statistic: {u_stat2:.1f}')
    print(f'p-value (one-sided, FP < NP): {mw_p2:.6f}')
    print(f'Result: {"Significant" if mw_p2 < 0.05 else "Not significant"} at alpha=0.05')
    
    pooled_std2 = np.sqrt((fp.std()**2 + np_grp.std()**2) / 2)
    cohens_d2 = (np_grp.mean() - fp.mean()) / pooled_std2
    print(f'Cohen\'s d: {cohens_d2:.3f} ({"small" if abs(cohens_d2) < 0.5 else "medium" if abs(cohens_d2) < 0.8 else "large"} effect)')

HYPOTHESIS 1: OWNERSHIP TYPE AND DOCUMENTATION GAP

--- Descriptive Statistics by Ownership ---
                       N   Mean  Median  Std Dev
ownership_category                              
Nonprofit           1925  49.48   49.38    10.78
Government           438  48.22   47.80    14.13
For-Profit           629  47.49   46.69    11.91
Other                 68  46.99   49.20     9.98

--- Kruskal-Wallis H-Test (all ownership groups) ---
H-statistic: 22.152
p-value: 0.000061
Result: Significant at alpha=0.05

--- One-Way ANOVA (all ownership groups) ---
F-statistic: 5.831
p-value: 0.000572
Result: Significant at alpha=0.05

--- Mann-Whitney U Test: For-Profit vs. Government ---
For-Profit mean: 47.49 (n=629)
Government mean: 48.22 (n=438)
U-statistic: 134140.5
p-value (one-sided, FP < Gov): 0.232989
Result: Not significant at alpha=0.05
Cohen's d: 0.055 (small effect)

--- Mann-Whitney U Test: For-Profit vs. Nonprofit ---
For-Profit mean: 47.49 (n=629)
Nonprofit mean: 49.48 (n=1925)


## Part 2: Hypothesis Testing — Provider Staffing Density (H2)

**H2**: Hospitals with higher provider-to-patient ratios have smaller documentation gaps.

We now test this with **direct staffing ratios** from the CMS POS file (extracted in NB10a):
- `physician_per_bed` — physicians per certified bed
- `clinical_staff_per_bed` — total clinical staff per bed
- `rn_per_bed` — registered nurses per bed
- `rn_skill_mix` — RN share of total nursing staff

Plus the original proxy: `resident_to_bed_ratio` from the IPPS impact file.

Tests:
- **Spearman rank correlation**: Non-parametric test for monotonic relationship
- **Pearson correlation**: Linear relationship test
- **Mann-Whitney U**: Teaching vs. non-teaching hospitals
- **Kruskal-Wallis**: Across teaching intensity levels (non, minor, major)

In [3]:
print('='*70)
print('HYPOTHESIS 2: PROVIDER DENSITY AND DOCUMENTATION GAP')
print('='*70)

# --- Direct Staffing Ratios from POS File (NB10a) ---
print('\n--- Staffing Ratios vs. Documentation Gap Score ---')
staffing_ratio_vars = [
    ('physician_per_bed', 'Physicians per Bed'),
    ('clinical_staff_per_bed', 'Clinical Staff per Bed'),
    ('rn_per_bed', 'RNs per Bed'),
    ('total_nursing_per_bed', 'Total Nursing per Bed'),
    ('rn_skill_mix', 'RN Skill Mix (RN/Total Nursing)'),
    ('resident_to_bed_ratio', 'Residents per Bed (IPPS)'),
]

h2_results = {}
print(f'\n{"Variable":<35s} {"Spearman r":>12s} {"p-value":>12s} {"Pearson r":>12s} {"p-value":>12s} {"N":>7s}')
print('-'*93)

for col, label in staffing_ratio_vars:
    if col in model_df.columns:
        temp = model_df.dropna(subset=[col, 'doc_gap_score'])
        if len(temp) >= 20:
            sp_r, sp_p = stats.spearmanr(temp[col], temp['doc_gap_score'])
            pe_r, pe_p = stats.pearsonr(temp[col], temp['doc_gap_score'])
            sig = '***' if sp_p < 0.001 else '**' if sp_p < 0.01 else '*' if sp_p < 0.05 else ''
            print(f'{label:<35s} {sp_r:>10.4f}{sig:>2s} {sp_p:>12.6f} {pe_r:>10.4f} {pe_p:>12.6f} {len(temp):>7,}')
            h2_results[col] = {'spearman_r': sp_r, 'spearman_p': sp_p, 'n': len(temp)}
        else:
            print(f'{label:<35s} {"insufficient data":>12s}')
    else:
        print(f'{label:<35s} {"not available":>12s}')

# --- Key test: physician_per_bed (most direct proxy for H2) ---
print('\n\n--- Primary H2 Test: Physician-per-Bed vs. Gap Score ---')
best_h2_col = None
for col in ['physician_per_bed', 'clinical_staff_per_bed', 'rn_per_bed', 'resident_to_bed_ratio']:
    if col in h2_results:
        best_h2_col = col
        break

if best_h2_col:
    res = h2_results[best_h2_col]
    direction = 'negative (supports H2 — more staff → lower gap)' if res['spearman_r'] < 0 else \
                'positive (contradicts H2 — more staff → higher gap)'
    print(f'Best available staffing measure: {best_h2_col}')
    print(f'Spearman r = {res["spearman_r"]:.4f}, p = {res["spearman_p"]:.6f}, N = {res["n"]:,}')
    print(f'Direction: {direction}')
    if res['spearman_r'] > 0 and res['spearman_p'] < 0.05:
        print('Note: Positive correlation may reflect that better-staffed hospitals treat')
        print('more complex patients, creating more documentation opportunities that go uncaptured.')

# --- Correlation: Resident-to-Bed Ratio vs. Gap Score (original proxy) ---
print('\n\n--- Resident-to-Bed Ratio vs. Documentation Gap Score (Original Proxy) ---')
ratio_df = model_df.dropna(subset=['resident_to_bed_ratio', 'doc_gap_score'])
print(f'Hospitals with resident-to-bed data: {len(ratio_df):,}')
print(f'Resident-to-bed ratio: mean={ratio_df["resident_to_bed_ratio"].mean():.4f}, '
      f'median={ratio_df["resident_to_bed_ratio"].median():.4f}')

spearman_r, spearman_p = stats.spearmanr(
    ratio_df['resident_to_bed_ratio'], ratio_df['doc_gap_score']
)
print(f'Spearman correlation: r={spearman_r:.4f}, p={spearman_p:.6f}')

pearson_r, pearson_p = stats.pearsonr(
    ratio_df['resident_to_bed_ratio'], ratio_df['doc_gap_score']
)
print(f'Pearson correlation: r={pearson_r:.4f}, p={pearson_p:.6f}')

# --- Teaching Status: Teaching vs. Non-Teaching ---
print('\n\n--- Teaching vs. Non-Teaching Hospitals ---')
teaching = model_df[model_df['is_teaching'] == True]['doc_gap_score'].values
non_teaching = model_df[model_df['is_teaching'] == False]['doc_gap_score'].values

print(f'Teaching mean: {teaching.mean():.2f} (n={len(teaching)})')
print(f'Non-teaching mean: {non_teaching.mean():.2f} (n={len(non_teaching)})')

u_teach, p_teach = stats.mannwhitneyu(teaching, non_teaching, alternative='two-sided')
print(f'Mann-Whitney U: {u_teach:.1f}, p={p_teach:.6f}')
print(f'Result: {"Significant" if p_teach < 0.05 else "Not significant"} at alpha=0.05')

pooled_teach = np.sqrt((teaching.std()**2 + non_teaching.std()**2) / 2)
d_teach = (teaching.mean() - non_teaching.mean()) / pooled_teach
print(f'Cohen\'s d: {d_teach:.3f} ({"small" if abs(d_teach) < 0.5 else "medium" if abs(d_teach) < 0.8 else "large"} effect)')

# --- Teaching Intensity Levels ---
print('\n\n--- Teaching Intensity Levels ---')
if 'teaching_intensity' in model_df.columns:
    intensity_stats = model_df.groupby('teaching_intensity')['doc_gap_score'].agg(
        ['count', 'mean', 'median', 'std']
    ).round(2)
    intensity_stats.columns = ['N', 'Mean', 'Median', 'Std Dev']
    print(intensity_stats)
    
    intensity_groups = [g['doc_gap_score'].dropna().values 
                        for _, g in model_df.groupby('teaching_intensity') 
                        if len(g) >= 5]
    h_int, p_int = stats.kruskal(*intensity_groups)
    print(f'\nKruskal-Wallis across intensity levels: H={h_int:.3f}, p={p_int:.6f}')
    print(f'Result: {"Significant" if p_int < 0.05 else "Not significant"} at alpha=0.05')

HYPOTHESIS 2: PROVIDER DENSITY AND DOCUMENTATION GAP

--- Staffing Ratios vs. Documentation Gap Score ---

Variable                              Spearman r      p-value    Pearson r      p-value       N
---------------------------------------------------------------------------------------------
Physicians per Bed                      0.0839***     0.000003    -0.0045     0.803608   3,060
Clinical Staff per Bed                 -0.0380 *     0.035326    -0.0096     0.595639   3,060
RNs per Bed                            -0.0230       0.203213    -0.0102     0.571057   3,060
Total Nursing per Bed                  -0.0268       0.138545    -0.0101     0.576636   3,060
RN Skill Mix (RN/Total Nursing)        -0.0276       0.130782    -0.0358     0.049615   3,003
Residents per Bed (IPPS)                0.1152***     0.000000    -0.0198     0.272743   3,060


--- Primary H2 Test: Physician-per-Bed vs. Gap Score ---
Best available staffing measure: physician_per_bed
Spearman r = 0.0839, p = 0.

## Part 3: Additional Predictor Analysis

Test associations between documentation gap score and other hospital characteristics:
- Hospital size (beds), urbanicity, wage index, DSH percentage, DRG diversity
- **NEW**: Quality metrics (star rating, quality score, mortality/safety/readmission rates)
- **NEW**: Staffing ratios (RN per bed, clinical staff per bed, RN skill mix)
- Census region

In [4]:
print('='*70)
print('ADDITIONAL PREDICTOR ANALYSIS')
print('='*70)

# --- Continuous predictors: Spearman correlations ---
print('\n--- Spearman Correlations with Documentation Gap Score ---')
continuous_vars = [
    # Original features
    ('beds', 'Hospital Size (Beds)'),
    ('wage_index', 'Wage Index'),
    ('dsh_pct', 'DSH Percentage (Safety-Net)'),
    ('drg_diversity', 'DRG Diversity'),
    ('resident_to_bed_ratio', 'Resident-to-Bed Ratio'),
    ('cmi', 'Case Mix Index'),
    # Staffing features from NB10a
    ('rn_per_bed', 'RN per Bed (POS)'),
    ('physician_per_bed', 'Physician per Bed (POS)'),
    ('clinical_staff_per_bed', 'Clinical Staff per Bed (POS)'),
    ('total_nursing_per_bed', 'Total Nursing per Bed (POS)'),
    ('rn_skill_mix', 'RN Skill Mix (POS)'),
    # Quality features from NB10a
    ('star_rating', 'CMS Star Rating'),
    ('quality_score', 'Quality Score (Composite)'),
    ('mort_pct_worse', 'Mortality % Worse'),
    ('safety_pct_worse', 'Safety % Worse'),
    ('readm_pct_worse', 'Readmission % Worse'),
]

print(f'\n{"Variable":<35s} {"r":>8s} {"p-value":>12s} {"Sig":>5s} {"N":>7s}')
print('-'*70)
for col, label in continuous_vars:
    if col in model_df.columns:
        temp = model_df.dropna(subset=[col, 'doc_gap_score'])
        if len(temp) >= 20:
            r, p = stats.spearmanr(temp[col], temp['doc_gap_score'])
            sig = ''
            if p < 0.05: sig = '*'
            if p < 0.01: sig = '**'
            if p < 0.001: sig = '***'
            print(f'{label:<35s} {r:>8.4f} {p:>12.6f} {sig:>5s} {len(temp):>7,}')
        else:
            print(f'{label:<35s} {"(n<20)":>8s}')

# --- Categorical: Urban vs. Rural ---
print('\n\n--- Urban vs. Rural ---')
urban = model_df[model_df['is_urban'] == True]['doc_gap_score'].values
rural = model_df[model_df['is_urban'] == False]['doc_gap_score'].values
print(f'Urban mean: {urban.mean():.2f} (n={len(urban)})')
print(f'Rural mean: {rural.mean():.2f} (n={len(rural)})')
u_ur, p_ur = stats.mannwhitneyu(urban, rural, alternative='two-sided')
print(f'Mann-Whitney U: {u_ur:.1f}, p={p_ur:.6f}')
print(f'Result: {"Significant" if p_ur < 0.05 else "Not significant"} at alpha=0.05')

# --- Quality tier analysis: Star Rating groups ---
print('\n\n--- Star Rating vs. Documentation Gap ---')
if 'star_rating' in model_df.columns:
    star_df = model_df.dropna(subset=['star_rating'])
    if len(star_df) >= 20:
        star_stats = star_df.groupby('star_rating')['doc_gap_score'].agg(
            ['count', 'mean', 'median', 'std']
        ).round(2)
        star_stats.columns = ['N', 'Mean', 'Median', 'Std Dev']
        print(star_stats)
        
        star_groups = [g['doc_gap_score'].dropna().values 
                       for _, g in star_df.groupby('star_rating') 
                       if len(g) >= 5]
        if len(star_groups) >= 2:
            h_star, p_star = stats.kruskal(*star_groups)
            print(f'\nKruskal-Wallis across star ratings: H={h_star:.3f}, p={p_star:.6f}')
            print(f'Result: {"Significant" if p_star < 0.05 else "Not significant"} at alpha=0.05')
    else:
        print('Insufficient data for star rating analysis')
else:
    print('Star rating not available in dataset')

# --- Census Region ---
print('\n\n--- Census Region ---')
if 'census_region' in model_df.columns:
    region_stats = model_df.groupby('census_region')['doc_gap_score'].agg(
        ['count', 'mean', 'median', 'std']
    ).round(2)
    region_stats.columns = ['N', 'Mean', 'Median', 'Std Dev']
    print(region_stats.sort_values('Mean', ascending=False))
    
    region_groups = [g['doc_gap_score'].dropna().values 
                     for _, g in model_df.groupby('census_region') 
                     if len(g) >= 5]
    h_reg, p_reg = stats.kruskal(*region_groups)
    print(f'\nKruskal-Wallis: H={h_reg:.3f}, p={p_reg:.6f}')
    print(f'Result: {"Significant" if p_reg < 0.05 else "Not significant"} at alpha=0.05')

ADDITIONAL PREDICTOR ANALYSIS

--- Spearman Correlations with Documentation Gap Score ---

Variable                                   r      p-value   Sig       N
----------------------------------------------------------------------
Hospital Size (Beds)                  0.2712     0.000000   ***   3,060
Wage Index                           -0.1448     0.000000   ***   3,060
DSH Percentage (Safety-Net)          -0.0575     0.001454    **   3,060
DRG Diversity                         0.4624     0.000000   ***   2,878
Resident-to-Bed Ratio                 0.1152     0.000000   ***   3,060
Case Mix Index                       -0.1848     0.000000   ***   3,060
RN per Bed (POS)                     -0.0230     0.203213         3,060
Physician per Bed (POS)               0.0839     0.000003   ***   3,060
Clinical Staff per Bed (POS)         -0.0380     0.035326     *   3,060
Total Nursing per Bed (POS)          -0.0268     0.138545         3,060
RN Skill Mix (POS)                   -0.0276  

Mortality % Worse                     0.1302     0.000000   ***   2,735
Safety % Worse                        0.1061     0.000000   ***   2,883
Readmission % Worse                   0.1106     0.000000   ***   2,994


--- Urban vs. Rural ---
Urban mean: 49.41 (n=2389)
Rural mean: 46.77 (n=671)
Mann-Whitney U: 933594.0, p=0.000000
Result: Significant at alpha=0.05


--- Star Rating vs. Documentation Gap ---
               N   Mean  Median  Std Dev
star_rating                             
1.0          211  48.54   49.53    10.92
2.0          606  48.79   49.46    10.31
3.0          831  48.75   49.27    10.37
4.0          672  47.90   47.97    10.23
5.0          216  47.03   47.06    10.03

Kruskal-Wallis across star ratings: H=6.394, p=0.171578
Result: Not significant at alpha=0.05


--- Census Region ---
                  N   Mean  Median  Std Dev
census_region                              
Northeast       441  51.83   52.43    10.09
South          1274  49.73   49.96    11.36
Midwest 

## Part 4: OLS Multiple Regression Model

We fit a multivariate OLS regression to identify which hospital characteristics independently predict higher documentation gap scores, controlling for confounders.

**Dependent variable**: `doc_gap_score` (continuous, 0–100)

**Independent variables**:
- *Structural*: ownership dummies, teaching status, log(beds), urban/rural
- *Financial*: wage index, DSH percentage
- *Clinical*: DRG diversity, resident-to-bed ratio
- *Staffing (NB10a)*: clinical staff per bed, RN skill mix, physician per bed
- *Quality (NB10a)*: star rating, readmission % worse, safety % worse

We fit two models: a **base model** (original features only) and an **enriched model** (with staffing + quality features) to show the incremental explanatory power of the new features.

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

def run_ols(reg_df, feature_cols, feature_labels, model_name='Model'):
    """Fit OLS regression and print results. Returns dict with key statistics."""
    X = reg_df[feature_cols].values
    y = reg_df['doc_gap_score'].values
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Fit OLS
    ols = LinearRegression()
    ols.fit(X_scaled, y)
    
    y_pred = ols.predict(X_scaled)
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r_squared = 1 - ss_res / ss_tot
    n, p = X_scaled.shape
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - p - 1)
    
    # Standard errors and p-values
    mse = ss_res / (n - p - 1)
    X_with_intercept = np.column_stack([np.ones(n), X_scaled])
    var_beta = mse * np.linalg.inv(X_with_intercept.T @ X_with_intercept).diagonal()
    se_beta = np.sqrt(var_beta)
    coefs_with_intercept = np.concatenate([[ols.intercept_], ols.coef_])
    t_stats = coefs_with_intercept / se_beta
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=n - p - 1)
    
    # F-statistic
    f_stat_reg = (ss_tot - ss_res) / p / (ss_res / (n - p - 1))
    f_p_value = stats.f.sf(f_stat_reg, p, n - p - 1)
    
    print(f'\n{"="*70}')
    print(f'{model_name} RESULTS')
    print(f'{"="*70}')
    print(f'R-squared:      {r_squared:.4f}')
    print(f'Adj R-squared:  {adj_r_squared:.4f}')
    print(f'F-statistic:    {f_stat_reg:.2f} (p={f_p_value:.6f})')
    print(f'Observations:   {n:,}')
    
    print(f'\n{"Variable":<35s} {"Coef":>8s} {"Std Err":>8s} {"t":>8s} {"p-value":>10s} {"Sig":>5s}')
    print('-'*78)
    print(f'{"Intercept":<35s} {coefs_with_intercept[0]:>8.3f} {se_beta[0]:>8.3f} {t_stats[0]:>8.3f} {p_values[0]:>10.6f}')
    
    for i, label in enumerate(feature_labels):
        coef = coefs_with_intercept[i+1]
        se = se_beta[i+1]
        t = t_stats[i+1]
        pv = p_values[i+1]
        sig = ''
        if pv < 0.05: sig = '*'
        if pv < 0.01: sig = '**'
        if pv < 0.001: sig = '***'
        print(f'{label:<35s} {coef:>8.3f} {se:>8.3f} {t:>8.3f} {pv:>10.6f} {sig:>5s}')
    
    print(f'\nNote: Coefficients are standardized (unit = 1 SD change in predictor).')
    print(f'Reference category for ownership: For-Profit')
    print(f'Significance: * p<0.05, ** p<0.01, *** p<0.001')
    
    return {
        'r_squared': r_squared, 'adj_r_squared': adj_r_squared,
        'f_stat': f_stat_reg, 'f_p': f_p_value, 'n': n, 'p': p,
        'coefs': coefs_with_intercept, 'se': se_beta, 't': t_stats, 'pvals': p_values,
        'ols': ols, 'scaler': scaler, 'feature_labels': feature_labels, 'feature_cols': feature_cols
    }


# ============================================================
# Prepare regression dataframe
# ============================================================
print('='*70)
print('OLS MULTIPLE REGRESSION: DOCUMENTATION GAP SCORE DRIVERS')
print('='*70)

# Base required columns
base_required = ['doc_gap_score', 'beds', 'wage_index', 'dsh_pct',
                 'resident_to_bed_ratio', 'ownership_category', 'is_teaching', 'is_urban']

reg_df = model_df.dropna(subset=base_required).copy()

# Feature engineering
reg_df['log_beds'] = np.log1p(reg_df['beds'])
reg_df['is_nonprofit'] = (reg_df['ownership_category'] == 'Nonprofit').astype(int)
reg_df['is_government'] = (reg_df['ownership_category'] == 'Government').astype(int)
reg_df['is_other'] = (reg_df['ownership_category'] == 'Other').astype(int)
reg_df['teaching'] = reg_df['is_teaching'].astype(int)
reg_df['urban'] = reg_df['is_urban'].astype(int)

if 'drg_diversity' in reg_df.columns:
    reg_df['drg_diversity_filled'] = reg_df['drg_diversity'].fillna(reg_df['drg_diversity'].median())
else:
    reg_df['drg_diversity_filled'] = 0

# ============================================================
# MODEL 1: Base Model (original features only)
# ============================================================
base_feature_cols = [
    'is_nonprofit', 'is_government', 'is_other',
    'teaching', 'resident_to_bed_ratio', 'log_beds',
    'urban', 'wage_index', 'dsh_pct', 'drg_diversity_filled',
]
base_feature_labels = [
    'Nonprofit (vs. For-Profit)', 'Government (vs. For-Profit)', 'Other (vs. For-Profit)',
    'Teaching Hospital', 'Resident-to-Bed Ratio', 'Log(Beds)',
    'Urban', 'Wage Index', 'DSH Percentage', 'DRG Diversity'
]

base_results = run_ols(reg_df, base_feature_cols, base_feature_labels, 'BASE MODEL')

# ============================================================
# MODEL 2: Enriched Model (with staffing + quality features)
# ============================================================
enriched_feature_cols = list(base_feature_cols)
enriched_feature_labels = list(base_feature_labels)

# Add staffing features if available (fill NaN with median)
staffing_additions = [
    ('clinical_staff_per_bed', 'Clinical Staff per Bed'),
    ('rn_skill_mix', 'RN Skill Mix'),
    ('physician_per_bed', 'Physician per Bed'),
]
for col, label in staffing_additions:
    if col in reg_df.columns and reg_df[col].notna().sum() >= 100:
        fill_col = f'{col}_filled'
        reg_df[fill_col] = reg_df[col].fillna(reg_df[col].median())
        enriched_feature_cols.append(fill_col)
        enriched_feature_labels.append(label)

# Add quality features if available
quality_additions = [
    ('star_rating', 'CMS Star Rating'),
    ('readm_pct_worse', 'Readmission % Worse'),
    ('safety_pct_worse', 'Safety % Worse'),
    ('quality_score', 'Quality Score (Composite)'),
]
for col, label in quality_additions:
    if col in reg_df.columns and reg_df[col].notna().sum() >= 100:
        fill_col = f'{col}_filled'
        reg_df[fill_col] = reg_df[col].fillna(reg_df[col].median())
        enriched_feature_cols.append(fill_col)
        enriched_feature_labels.append(label)

if len(enriched_feature_cols) > len(base_feature_cols):
    enriched_results = run_ols(reg_df, enriched_feature_cols, enriched_feature_labels, 'ENRICHED MODEL')
    
    # Compare models
    delta_r2 = enriched_results['r_squared'] - base_results['r_squared']
    delta_adj_r2 = enriched_results['adj_r_squared'] - base_results['adj_r_squared']
    print(f'\n{"="*70}')
    print(f'MODEL COMPARISON')
    print(f'{"="*70}')
    print(f'Base model R²:     {base_results["r_squared"]:.4f} (Adj: {base_results["adj_r_squared"]:.4f})')
    print(f'Enriched model R²: {enriched_results["r_squared"]:.4f} (Adj: {enriched_results["adj_r_squared"]:.4f})')
    print(f'Improvement:       +{delta_r2:.4f} R² (+{delta_adj_r2:.4f} Adj R²)')
    print(f'New features added {len(enriched_feature_cols) - len(base_feature_cols)} predictors from NB10a')
    
    # Use enriched model for downstream analysis
    final_results = enriched_results
else:
    print('\nNo enriched features available — using base model only.')
    final_results = base_results

# Store for Part 6 summary
r_squared = final_results['r_squared']
adj_r_squared = final_results['adj_r_squared']
ols = final_results['ols']
f_stat_reg = final_results['f_stat']
f_p_value = final_results['f_p']
final_feature_labels = final_results['feature_labels']
final_coefs = final_results['coefs']
final_se = final_results['se']
final_t = final_results['t']
final_pvals = final_results['pvals']

OLS MULTIPLE REGRESSION: DOCUMENTATION GAP SCORE DRIVERS

BASE MODEL RESULTS
R-squared:      0.1807
Adj R-squared:  0.1780
F-statistic:    67.24 (p=0.000000)
Observations:   3,060

Variable                                Coef  Std Err        t    p-value   Sig
------------------------------------------------------------------------------
Intercept                             48.835    0.189  257.739   0.000000
Nonprofit (vs. For-Profit)             0.482    0.240    2.009   0.044661     *
Government (vs. For-Profit)            0.287    0.234    1.230   0.218633      
Other (vs. For-Profit)                 0.095    0.205    0.463   0.643264      
Teaching Hospital                      0.955    0.225    4.239   0.000023   ***
Resident-to-Bed Ratio                 -1.976    0.227   -8.694   0.000000   ***
Log(Beds)                             -0.005    0.300   -0.018   0.985958      
Urban                                  0.645    0.213    3.021   0.002541    **
Wage Index                

## Part 5: Logistic Regression — Predicting High-Gap Hospitals

Binary outcome: Is the hospital classified as "High" documentation gap (score >= 65)?

This complements the OLS model by asking: which characteristics make a hospital most likely to fall into the high-risk category?

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

print('='*70)
print('LOGISTIC REGRESSION: PREDICTING HIGH-GAP HOSPITALS')
print('='*70)

# Create binary outcome
reg_df['is_high_gap'] = (reg_df['doc_gap_score'] >= 65).astype(int)

print(f'\nHigh-gap hospitals: {reg_df["is_high_gap"].sum():,} ({100*reg_df["is_high_gap"].mean():.1f}%)')
print(f'Non-high-gap hospitals: {(1-reg_df["is_high_gap"]).sum():.0f} ({100*(1-reg_df["is_high_gap"].mean()):.1f}%)')

# Use same enriched feature set as OLS
logit_feature_cols = final_results['feature_cols']
logit_feature_labels = final_results['feature_labels']

X_logit = reg_df[logit_feature_cols].values
y_binary = reg_df['is_high_gap'].values

# Standardize
scaler_logit = StandardScaler()
X_logit_scaled = scaler_logit.fit_transform(X_logit)

# Fit logistic regression
logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_logit_scaled, y_binary)

y_pred_proba = logit.predict_proba(X_logit_scaled)[:, 1]
y_pred_class = logit.predict(X_logit_scaled)

# AUC-ROC
auc = roc_auc_score(y_binary, y_pred_proba)
print(f'\nAUC-ROC: {auc:.4f}')

# Coefficients as odds ratios
print(f'\n{"Variable":<35s} {"Coef":>8s} {"Odds Ratio":>12s} {"Direction":>12s}')
print('-'*70)
for i, label in enumerate(logit_feature_labels):
    coef = logit.coef_[0][i]
    odds_ratio = np.exp(coef)
    direction = '↑ Risk' if coef > 0 else '↓ Risk'
    print(f'{label:<35s} {coef:>8.3f} {odds_ratio:>12.3f} {direction:>12s}')

# Top risk and protective factors
print(f'\n--- Top 5 Risk Factors (increase odds of high gap) ---')
risk_pairs = sorted(zip(logit_feature_labels, logit.coef_[0]), key=lambda x: x[1], reverse=True)
for i, (label, coef) in enumerate(risk_pairs[:5], 1):
    if coef > 0:
        print(f'  {i}. {label}: OR={np.exp(coef):.3f} (+{100*(np.exp(coef)-1):.1f}% higher odds per 1 SD)')

print(f'\n--- Top 5 Protective Factors (decrease odds of high gap) ---')
for i, (label, coef) in enumerate(reversed(risk_pairs[-5:]), 1):
    if coef < 0:
        print(f'  {i}. {label}: OR={np.exp(coef):.3f} ({100*(np.exp(coef)-1):.1f}% lower odds per 1 SD)')

print(f'\nInterpretation: Odds ratio > 1 means higher odds of being a high-gap hospital.')
print(f'Coefficients are based on standardized predictors (1 SD change).')

LOGISTIC REGRESSION: PREDICTING HIGH-GAP HOSPITALS

High-gap hospitals: 277 (9.1%)
Non-high-gap hospitals: 2783 (90.9%)

AUC-ROC: 0.7533

Variable                                Coef   Odds Ratio    Direction
----------------------------------------------------------------------
Nonprofit (vs. For-Profit)             0.029        1.030       ↑ Risk
Government (vs. For-Profit)            0.156        1.169       ↑ Risk
Other (vs. For-Profit)                -0.241        0.786       ↓ Risk
Teaching Hospital                      0.119        1.126       ↑ Risk
Resident-to-Bed Ratio                 -0.398        0.672       ↓ Risk
Log(Beds)                             -0.723        0.486       ↓ Risk
Urban                                  0.095        1.100       ↑ Risk
Wage Index                            -0.706        0.493       ↓ Risk
DSH Percentage                         0.523        1.687       ↑ Risk
DRG Diversity                          0.916        2.499       ↑ Risk
Clinical S

## Part 6: Summary of Findings

In [7]:
print('='*70)
print('SUMMARY OF STATISTICAL FINDINGS')
print('='*70)

print('''
HYPOTHESIS 1 — Ownership Type
------------------------------
H1 predicted that for-profit hospitals would have smaller documentation
gaps than government/public hospitals due to better resourcing.
''')
if 'For-Profit' in groups and 'Government' in groups:
    fp_mean = groups['For-Profit'].mean()
    gov_mean = groups['Government'].mean()
    direction_h1 = 'SUPPORTED' if fp_mean < gov_mean and mw_p < 0.05 else \
                   'PARTIALLY SUPPORTED' if fp_mean < gov_mean else 'NOT SUPPORTED'
    print(f'  For-Profit mean gap score: {fp_mean:.2f}')
    print(f'  Government mean gap score: {gov_mean:.2f}')
    print(f'  Difference: {gov_mean - fp_mean:.2f} points')
    print(f'  Mann-Whitney p-value: {mw_p:.6f}')
    print(f'  Verdict: {direction_h1}')

print('''
HYPOTHESIS 2 — Provider Staffing Density
-----------------------------------------
H2 predicted that hospitals with greater provider-to-patient ratios
would have smaller documentation gaps.
''')

# Report best staffing measure results
if h2_results:
    print('  Staffing ratio correlations with gap score:')
    for col, res in h2_results.items():
        label = col.replace('_', ' ').title()
        sig = '***' if res['spearman_p'] < 0.001 else '**' if res['spearman_p'] < 0.01 else \
              '*' if res['spearman_p'] < 0.05 else 'ns'
        direction = 'lower gap' if res['spearman_r'] < 0 else 'higher gap'
        print(f'    {label}: r={res["spearman_r"]:.4f} ({sig}) → more staff → {direction}')

# Determine overall H2 verdict
h2_negative_sigs = sum(1 for r in h2_results.values() if r['spearman_r'] < 0 and r['spearman_p'] < 0.05)
h2_positive_sigs = sum(1 for r in h2_results.values() if r['spearman_r'] > 0 and r['spearman_p'] < 0.05)
if h2_negative_sigs > h2_positive_sigs:
    h2_verdict = 'SUPPORTED'
elif h2_positive_sigs > h2_negative_sigs:
    h2_verdict = 'CONTRADICTED'
else:
    h2_verdict = 'MIXED / NOT SUPPORTED'

print(f'\n  Significant negative correlations: {h2_negative_sigs}')
print(f'  Significant positive correlations: {h2_positive_sigs}')
print(f'  Verdict: {h2_verdict}')

if h2_positive_sigs > 0:
    print('  Note: Positive correlations may reflect that better-staffed hospitals')
    print('  treat more complex patients, creating more documentation opportunities')
    print('  that go uncaptured — not necessarily worse documentation practices.')

# Teaching hospital results
print(f'\n  Teaching vs Non-Teaching:')
print(f'  Teaching mean: {teaching.mean():.2f} vs Non-teaching: {non_teaching.mean():.2f}')
print(f'  Mann-Whitney p={p_teach:.6f}')

print(f'''
REGRESSION MODEL
----------------
  R-squared: {r_squared:.4f} (Adj: {adj_r_squared:.4f})
  The model explains {100*r_squared:.1f}% of variance in documentation gap scores.
  
  Strongest predictors (by standardized coefficient magnitude):
''')

# Sort by absolute coefficient
coef_pairs = sorted(zip(final_feature_labels, ols.coef_), key=lambda x: abs(x[1]), reverse=True)
for i, (label, coef) in enumerate(coef_pairs[:7], 1):
    direction = 'increases' if coef > 0 else 'decreases'
    print(f'  {i}. {label}: 1 SD increase {direction} gap score by {abs(coef):.2f} points')

print(f'''
LOGISTIC REGRESSION (High-Gap Prediction)
------------------------------------------
  AUC-ROC: {auc:.4f}
  Predicting whether a hospital falls in the "High" gap tier (score >= 65).
''')

# Save results
output_dir = PROJECT_ROOT / 'data' / 'outputs' / 'nb10_statistical_model'
output_dir.mkdir(parents=True, exist_ok=True)

# Save regression coefficients (enriched model)
coef_df = pd.DataFrame({
    'variable': ['Intercept'] + list(final_feature_labels),
    'coefficient': final_coefs,
    'std_error': final_se,
    't_statistic': final_t,
    'p_value': final_pvals
})
coef_df.to_csv(output_dir / 'regression_coefficients.csv', index=False)

# Save logistic regression odds ratios
logit_df = pd.DataFrame({
    'variable': list(logit_feature_labels),
    'coefficient': logit.coef_[0],
    'odds_ratio': np.exp(logit.coef_[0])
})
logit_df.to_csv(output_dir / 'logistic_odds_ratios.csv', index=False)

# Save hypothesis test results
hypothesis_results = {
    'h1_fp_mean': fp_mean if 'For-Profit' in groups else None,
    'h1_gov_mean': gov_mean if 'Government' in groups else None,
    'h1_mw_p': mw_p if 'For-Profit' in groups and 'Government' in groups else None,
    'h1_verdict': direction_h1,
    'h2_verdict': h2_verdict,
    'h2_negative_sigs': h2_negative_sigs,
    'h2_positive_sigs': h2_positive_sigs,
    'ols_r_squared': r_squared,
    'ols_adj_r_squared': adj_r_squared,
    'logit_auc': auc,
}
pd.Series(hypothesis_results).to_csv(output_dir / 'hypothesis_results.csv')

print(f'\nResults saved to: {output_dir}')
print(f'  - regression_coefficients.csv')
print(f'  - logistic_odds_ratios.csv')
print(f'  - hypothesis_results.csv')

SUMMARY OF STATISTICAL FINDINGS

HYPOTHESIS 1 — Ownership Type
------------------------------
H1 predicted that for-profit hospitals would have smaller documentation
gaps than government/public hospitals due to better resourcing.

  For-Profit mean gap score: 47.49
  Government mean gap score: 48.22
  Difference: 0.72 points
  Mann-Whitney p-value: 0.232989
  Verdict: PARTIALLY SUPPORTED

HYPOTHESIS 2 — Provider Staffing Density
-----------------------------------------
H2 predicted that hospitals with greater provider-to-patient ratios
would have smaller documentation gaps.

  Staffing ratio correlations with gap score:
    Physician Per Bed: r=0.0839 (***) → more staff → higher gap
    Clinical Staff Per Bed: r=-0.0380 (*) → more staff → lower gap
    Rn Per Bed: r=-0.0230 (ns) → more staff → lower gap
    Total Nursing Per Bed: r=-0.0268 (ns) → more staff → lower gap
    Rn Skill Mix: r=-0.0276 (ns) → more staff → lower gap
    Resident To Bed Ratio: r=0.1152 (***) → more staff → hi

## Part 7: ANOVA — Case Mix Index by Ownership Type

**Research Question**: Does Case Mix Index differ significantly across hospital ownership categories (Government, For-Profit, Nonprofit, Other)?

This analysis supports Figure 1 in the blog post by testing whether the observed differences in Case Mix Index across ownership types are statistically significant.

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

# Load the feature-engineered hospital dataset
features_path = Path('../../data/outputs/nb05_features/hospital_features.csv')
df_features = pd.read_csv(features_path)

# ─────────────────────────────────────────────────────────────
# Descriptive Statistics
# ─────────────────────────────────────────────────────────────
print('=' * 70)
print('ANOVA: CASE MIX INDEX BY OWNERSHIP TYPE')
print('=' * 70)

group_stats = df_features.groupby('ownership_category')['cmi'].agg(['count', 'mean', 'median', 'std'])
group_stats.columns = ['N', 'Mean CMI', 'Median CMI', 'Std Dev']
group_stats = group_stats.sort_values('Mean CMI', ascending=False)
print('\nDescriptive Statistics:')
print(group_stats.round(4).to_string())

# ─────────────────────────────────────────────────────────────
# One-Way ANOVA
# ─────────────────────────────────────────────────────────────
groups = [g['cmi'].dropna().values for _, g in df_features.groupby('ownership_category')]
group_names = [name for name, _ in df_features.groupby('ownership_category')]

F_stat, p_anova = stats.f_oneway(*groups)

print(f'\nOne-Way ANOVA:')
print(f'  F({len(groups)-1}, {sum(len(g) for g in groups)-len(groups)}) = {F_stat:.4f}')
print(f'  p-value = {p_anova:.2e}')
print(f'  Result: {"SIGNIFICANT" if p_anova < 0.05 else "NOT SIGNIFICANT"} at α = 0.05')

# ─────────────────────────────────────────────────────────────
# Levene's Test for Homogeneity of Variances
# ─────────────────────────────────────────────────────────────
levene_stat, levene_p = stats.levene(*groups)
print(f'\nLevene\'s Test (equal variances assumption):')
print(f'  W = {levene_stat:.4f}, p = {levene_p:.4e}')
print(f'  Variances are {"NOT equal" if levene_p < 0.05 else "equal"} (α = 0.05)')

# Since variances are likely unequal, also run Welch's ANOVA (Kruskal-Wallis as nonparametric alternative)
H_stat, p_kruskal = stats.kruskal(*groups)
print(f'\nKruskal-Wallis H Test (non-parametric alternative):')
print(f'  H = {H_stat:.4f}, p = {p_kruskal:.2e}')

# ─────────────────────────────────────────────────────────────
# Post-Hoc Pairwise Comparisons (Bonferroni corrected)
# ─────────────────────────────────────────────────────────────
print(f'\nPost-Hoc Pairwise t-tests (Bonferroni corrected):')
print('-' * 70)
print(f'{"Comparison":<35} {"Diff":>8} {"t-stat":>8} {"p (raw)":>12} {"p (Bonf)":>12} {"Sig":>5}')
print('-' * 70)

n_comparisons = len(list(combinations(range(len(groups)), 2)))

for i, j in combinations(range(len(groups)), 2):
    t_stat, p_raw = stats.ttest_ind(groups[i], groups[j], equal_var=False)  # Welch's t-test
    p_bonf = min(p_raw * n_comparisons, 1.0)
    diff = np.mean(groups[i]) - np.mean(groups[j])
    sig = '***' if p_bonf < 0.001 else '**' if p_bonf < 0.01 else '*' if p_bonf < 0.05 else 'ns'
    label = f'{group_names[i]} vs {group_names[j]}'
    print(f'{label:<35} {diff:>+8.4f} {t_stat:>8.3f} {p_raw:>12.2e} {p_bonf:>12.2e} {sig:>5}')

print('-' * 70)
print('Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant')

# ─────────────────────────────────────────────────────────────
# Effect Size (Eta-squared)
# ─────────────────────────────────────────────────────────────
grand_mean = np.mean(np.concatenate(groups))
ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
ss_total = sum(np.sum((g - grand_mean)**2) for g in groups)
eta_squared = ss_between / ss_total
print(f'\nEffect Size:')
print(f'  η² (eta-squared) = {eta_squared:.4f}')
print(f'  Interpretation: {"Large" if eta_squared >= 0.14 else "Medium" if eta_squared >= 0.06 else "Small"} effect')

print(f'\n{"=" * 70}')
print('CONCLUSION: Case Mix Index differs significantly across all ownership')
print('categories (F = {:.1f}, p < 0.001). The "Other" category has the highest'.format(F_stat))
print('mean CMI (2.57), followed by For-Profit (1.79), Nonprofit (1.76), and')
print('Government (1.63). All pairwise comparisons are significant at p < 0.001')
print('after Bonferroni correction.')
print('=' * 70)

ANOVA: CASE MIX INDEX BY OWNERSHIP TYPE

Descriptive Statistics:
                       N  Mean CMI  Median CMI  Std Dev
ownership_category                                     
Other                 68    2.5702      2.5042   0.6158
For-Profit           629    1.7940      1.7081   0.4934
Nonprofit           1925    1.7641      1.7117   0.3578
Government           438    1.6262      1.5788   0.4079

One-Way ANOVA:
  F(3, 3056) = 108.6192
  p-value = 8.00e-67
  Result: SIGNIFICANT at α = 0.05

Levene's Test (equal variances assumption):
  W = 22.0335, p = 4.1061e-14
  Variances are NOT equal (α = 0.05)

Kruskal-Wallis H Test (non-parametric alternative):
  H = 175.4470, p = 8.48e-38

Post-Hoc Pairwise t-tests (Bonferroni corrected):
----------------------------------------------------------------------
Comparison                              Diff   t-stat      p (raw)     p (Bonf)   Sig
----------------------------------------------------------------------
For-Profit vs Government       